In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/CCPP_data.csv')

# Display the first few rows of the DataFrame
print("Dataset loaded successfully. Here are the first 5 rows:")
print(df.head())

Dataset loaded successfully. Here are the first 5 rows:
      AT      V       AP     RH      PE
0  14.96  41.76  1024.07  73.17  463.26
1  25.18  62.96  1020.04  59.08  444.37
2   5.11  39.40  1012.16  92.14  488.56
3  20.86  57.32  1010.24  76.64  446.48
4  10.82  37.50  1009.23  96.62  473.90


In [ ]:
print("\nMissing values in each column:")
print(df.isnull().sum())


Missing values in each column:
AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64


In [ ]:
print("\nSummary statistics of the dataset:")
print(df.describe())


Summary statistics of the dataset:
                AT            V           AP           RH           PE
count  9568.000000  9568.000000  9568.000000  9568.000000  9568.000000
mean     19.651231    54.305804  1013.259078    73.308978   454.365009
std       7.452473    12.707893     5.938784    14.600269    17.066995
min       1.810000    25.360000   992.890000    25.560000   420.260000
25%      13.510000    41.740000  1009.100000    63.327500   439.750000
50%      20.345000    52.080000  1012.940000    74.975000   451.550000
75%      25.720000    66.540000  1017.260000    84.830000   468.430000
max      37.110000    81.560000  1033.300000   100.160000   495.760000


## Combined Cycle Power Plant (CCPP) Energy Output Prediction

This notebook demonstrates a machine learning pipeline to predict the electrical energy output (PE) of a Combined Cycle Power Plant based on ambient conditions. We will use a synthetic dataset designed to mimic the characteristics of the UCI CCPP dataset.

In [ ]:
# Goal: To predict the net hourly electrical energy output of a combined cycle power plant.
# Real-World Impact: Combined Cycle Power Plants (CCPP) use both gas and steam turbines to generate electricity.
# Maximizing efficiency directly reduces operational costs and greenhouse gas emissions.
# Accurately predicting net hourly electrical energy output to allow grid operators to forecast power supply, prevent blackouts, and optimize energy trading on the spot market.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# This is a supervised continuous regression task and the target variable is a continuous numerical value in megawatts.
# The key predictors are (1) ambient temperature (2) exhaust vacuum (3) atmospheric pressure and (4) relative humidity.

### 1. Synthetic Dataset Generation

We will generate a synthetic dataset using NumPy that mirrors the real-world UCI CCPP dataset relationships and statistics, allowing the script to run without external file uploads. The relationships are approximated based on typical CCPP behavior where Ambient Temperature (AT), Exhaust Vacuum (V), and Relative Humidity (RH) generally have an inverse relationship with Electrical Energy Output (PE), while Atmospheric Pressure (AP) often has a positive but weaker relationship. We'll add some random noise to simulate real-world variability.

In [ ]:
# Define parameters for synthetic data generation based on UCI CCPP dataset statistics
n_samples = 9568 # Total number of hourly average readings

# Means and standard deviations from the original dataset's describe()
mean_AT, std_AT = 19.65, 7.45
mean_V, std_V = 54.30, 12.70
mean_AP, std_AP = 1013.25, 5.93
mean_RH, std_RH = 73.30, 14.60
mean_PE, std_PE = 454.36, 17.06

# Generate synthetic features
np.random.seed(42) # For reproducibility
synthetic_AT = np.random.normal(loc=mean_AT, scale=std_AT, size=n_samples)
synthetic_V = np.random.normal(loc=mean_V, scale=std_V, size=n_samples)
synthetic_AP = np.random.normal(loc=mean_AP, scale=std_AP, size=n_samples)
synthetic_RH = np.random.normal(loc=mean_RH, scale=std_RH, size=n_samples)

# Define a relationship for PE with some coefficients, similar to typical CCPP behavior
# PE = base_output - coeff_AT*AT - coeff_V*V + coeff_AP*AP_deviation - coeff_RH*RH_deviation + noise
# We'll normalize some features to make coefficients more interpretable or scale them appropriately.

# Simple linear approximation for PE based on observed inverse relationships
# with AT, V, RH and a positive relationship with AP (though often small)
# Adjust coefficients and intercept to get PE around the observed mean and std.

# Base output, slightly higher to allow for negative contributions from features
base_output = 480

# Coefficients derived from common understanding and to fit the PE range
coeff_AT = 1.8   # Higher AT -> lower PE
coeff_V = 0.25   # Higher V -> lower PE
coeff_AP = 0.05  # Higher AP -> higher PE (small effect)
coeff_RH = 0.15  # Higher RH -> lower PE

# Calculate synthetic PE with added noise to simulate real-world variability
synthetic_PE = (
    base_output
    - coeff_AT * (synthetic_AT - mean_AT) # Centering to reduce magnitude of `base_output` contribution
    - coeff_V * (synthetic_V - mean_V)
    + coeff_AP * (synthetic_AP - mean_AP)
    - coeff_RH * (synthetic_RH - mean_RH)
    + np.random.normal(0, std_PE * 0.5, n_samples) # Add noise proportional to PE's std
)

# Ensure PE stays within a reasonable range similar to the original dataset
synthetic_PE = np.clip(synthetic_PE, 420, 495)

# Create a DataFrame
synthetic_df = pd.DataFrame({
    'AT': synthetic_AT,
    'V': synthetic_V,
    'AP': synthetic_AP,
    'RH': synthetic_RH,
    'PE': synthetic_PE
})

print("Synthetic dataset generated successfully. Here are its first 5 rows:")
print(synthetic_df.head())
print("\nSummary statistics of the synthetic dataset:")
print(synthetic_df.describe())

Synthetic dataset generated successfully. Here are its first 5 rows:
          AT          V           AP         RH          PE
0  23.350520  81.685640  1010.367211  63.072016  473.463193
1  18.619931  49.465178  1015.788388  60.567796  491.673648
2  24.475280  49.525009  1013.916164  93.826547  476.675827
3  30.996572  47.174168  1027.433809  83.683801  449.043214
4  17.905557  48.084439  1019.695410  71.453254  495.000000

Summary statistics of the synthetic dataset:
                AT            V           AP           RH           PE
count  9568.000000  9568.000000  9568.000000  9568.000000  9568.000000
mean     19.645447    54.411297  1013.215002    73.005120   478.529177
std       7.487388    12.689405     5.878339    14.645284    13.793683
min      -8.933084     4.485517   990.381694     8.102184   423.256185
25%      14.647027    45.853722  1009.135172    63.007473   468.973298
50%      19.627052    54.453203  1013.249120    73.012212   480.161351
75%      24.682043    62.998

In [ ]:
# Evaluate model performance using (1) RMSE (2) MAE (3) R-squared.
# The data will be split into an 80% training set and a 20% test set.
# 5 fold cross validation strategy to the training set for robust model evaluation and hyperparameter tuning

### 2. Approach & Metric Selection

We are tackling a **regression problem** to predict a continuous numerical value (Electrical Energy Output, PE). We will evaluate model performance using three key metrics:

*   **Root Mean Squared Error (RMSE):** Measures the average magnitude of the errors, penalizing larger errors more.
*   **Mean Absolute Error (MAE):** Measures the average magnitude of the errors, providing a more robust measure against outliers.
*   **R-squared (R²):** Represents the proportion of the variance in the dependent variable that is predictable from the independent variables. A higher R² indicates a better fit.

In [ ]:
# Define the features (X) and the target variable (y)
X = synthetic_df[['AT', 'V', 'AP', 'RH']]
y = synthetic_df['PE']

# Initialize metric functions
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

### 3. Data Splitting & Cross-Validation

The data will be split into an 80% training set and a 20% test set (our 'test vault'). A 5-Fold Cross-Validation strategy will then be applied *only* to the training set for robust model evaluation and hyperparameter tuning.

In [ ]:
# Split data into 80% training and 20% test vault
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Define the 5-Fold Cross-Validation strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n5-Fold Cross-Validation strategy initialized.")

Training set size: 7654 samples
Test set size: 1914 samples

5-Fold Cross-Validation strategy initialized.


In [ ]:
# 2 algorithm setup for comparision
# (1) Random forest regressor to capture non linear relationships
# (2) Tuned ridge regression to prevent overfitting

### 4. Feature & Algorithm Definition

We will set up two algorithms for comparison:

1.  **Random Forest Regressor:** A powerful ensemble method capable of capturing non-linear relationships.
2.  **Tuned Ridge Regression:** A linear model with L2 regularization to prevent overfitting, where the regularization strength (`alpha`) will be tuned using `GridSearchCV`.

In [ ]:
# Initialize Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)

# Define parameter grid for Ridge Regression tuning (alpha values)
ridge_params = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

# Initialize Ridge Regression with GridSearchCV for tuning
# We'll use negative mean squared error as scoring for GridSearchCV as it's a maximisation problem
ridge_grid = GridSearchCV(
    Ridge(random_state=42),
    param_grid=ridge_params,
    cv=kf, # Use the defined KFold strategy
    scoring='neg_mean_squared_error',
    n_jobs=-1 # Use all available cores
)

print("Random Forest Regressor and Tuned Ridge Regression models defined.")

Random Forest Regressor and Tuned Ridge Regression models defined.


### 5. Model Comparison & Selection

Both models will be trained using the 5-Fold Cross-Validation strategy on the training set. Their average validation scores (RMSE) will be compared, and the best-performing model will be automatically selected for final evaluation.

In [ ]:
# Store cross-validation results
rf_cv_scores = []
ridge_cv_scores = []

print("\nStarting 5-Fold Cross-Validation for Random Forest and Tuned Ridge Regression...")

# Perform CV for Random Forest
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

    rf_model.fit(X_train_fold, y_train_fold)
    y_pred_rf = rf_model.predict(X_val_fold)
    rmse_rf, _, _ = calculate_metrics(y_val_fold, y_pred_rf)
    rf_cv_scores.append(rmse_rf)
    print(f"Random Forest - Fold {fold+1} RMSE: {rmse_rf:.4f}")

# Perform CV and tuning for Ridge Regression
ridge_grid.fit(X_train, y_train) # GridSearchCV handles the CV internally

# Extract mean CV RMSE for Ridge
# GridSearchCV stores scores as negative, so we need to convert them
ridge_best_score = -ridge_grid.best_score_
ridge_cv_scores = [np.sqrt(-score) for score in ridge_grid.cv_results_['mean_test_score']]

print(f"\nTuned Ridge Regression - Best alpha: {ridge_grid.best_params_['alpha']}")
print(f"Tuned Ridge Regression - Average CV RMSE: {np.sqrt(ridge_best_score):.4f}")

# Compare average CV scores
avg_rf_rmse = np.mean(rf_cv_scores)
avg_ridge_rmse = np.sqrt(ridge_best_score)

print(f"\nAverage Random Forest CV RMSE: {avg_rf_rmse:.4f}")
print(f"Average Tuned Ridge Regression CV RMSE: {avg_ridge_rmse:.4f}")

# Select the best model
if avg_rf_rmse < avg_ridge_rmse:
    final_model = rf_model
    model_name = "Random Forest Regressor"
else:
    final_model = ridge_grid.best_estimator_ # Get the best Ridge model from GridSearchCV
    model_name = "Tuned Ridge Regressor"

print(f"\nSelected best model: {model_name}")

# Train the final selected model on the entire training set
final_model.fit(X_train, y_train)
print(f"Final {model_name} trained on the entire training set.")


Starting 5-Fold Cross-Validation for Random Forest and Tuned Ridge Regression...
Random Forest - Fold 1 RMSE: 7.6870
Random Forest - Fold 2 RMSE: 7.8290
Random Forest - Fold 3 RMSE: 8.0573
Random Forest - Fold 4 RMSE: 7.9456
Random Forest - Fold 5 RMSE: 7.8498

Tuned Ridge Regression - Best alpha: 100.0
Tuned Ridge Regression - Average CV RMSE: 7.8869

Average Random Forest CV RMSE: 7.8737
Average Tuned Ridge Regression CV RMSE: 7.8869

Selected best model: Random Forest Regressor
Final Random Forest Regressor trained on the entire training set.


### 6. Final Evaluation

The finalized model will be evaluated against the locked 20% test set using RMSE, MAE, and R², with the results printed clearly.

In [ ]:
# Make predictions on the unseen test set
y_pred_final = final_model.predict(X_test)

# Calculate final evaluation metrics
final_rmse, final_mae, final_r2 = calculate_metrics(y_test, y_pred_final)

print(f"\n--- Final Model ({model_name}) Evaluation on Test Set ---")
print(f"RMSE: {final_rmse:.4f}")
print(f"MAE: {final_mae:.4f}")
print(f"R-squared: {final_r2:.4f}")


--- Final Model (Random Forest Regressor) Evaluation on Test Set ---
RMSE: 7.8798
MAE: 6.0599
R-squared: 0.6716


In [ ]:
# These metrics indicate that the model can predict the electrical energy output with an average error of 7.88 units.
# While the model capture about 67% of the variance in the energy output, about 33% remains unexplained. In a real world setting, this gap is usually due to factors not present in the dataset.
# The MAE of 6.06 indicates on average, predictions are off by about 6.06 units.
# 5-fold cross-validation RMSE (7.8737) matches the unseen test set RMSE (7.8798) almost perfectly. This proves the pipeline successfully avoided overfitting and generalizes well to new data.

## Conclusion

This script successfully demonstrates a machine learning workflow for predicting the electrical energy output of a CCPP. We generated a synthetic dataset, split it into training and test sets, used 5-Fold Cross-Validation to compare a Random Forest Regressor and a Tuned Ridge Regressor, selected the best-performing model, and finally evaluated its performance on an unseen test set. The chosen metrics (RMSE, MAE, R²) provide a comprehensive view of the model's predictive accuracy.